# ارزیابی جست‌وجوی بصری

**پرسش:** وقتی خریدار از فرشی عکس می‌گیرد، همان فرش چندم می‌آید؟

**مجموعه‌ی پرسش، دیده‌نشده به‌طور ساختاری.** در `ingest_catalog.py` فقط تصویر
`flat` هر فرش امبد می‌شود، پس سه تصویر دیگرِ همان فرش — `cover`، `room`،
`gallery` — تصویرهایی هستند که ایندکس هرگز ندیده است. برچسب هم لازم ندارند:
نام فایل، اسلاگ فرش را با خود دارد.

**۱۲۰ پرسش، ۴۰ فرش، یک پاسخ درست برای هر پرسش.** با یک پاسخ درست،
Precision@k به‌طور مکانیکی همان hit@k/k است و اطلاعات تازه‌ای ندارد؛ پس عدد
اصلی «نرخ اصابت» است و کنارش MRR، که تنها عددی است که تفاوت رتبه‌ی ۲ و ۲۰ را
می‌بیند.

In [ ]:
import sys
from pathlib import Path

# The notebooks live beside the backend, not inside it, so that `app` is
# importable exactly the way the server imports it — same package, same module
# state, no copy.
sys.path.insert(0, str(Path.cwd().parent / "backend"))

In [ ]:
import asyncio

from app.db.session import SessionLocal
from app.eval import retrieval
from app.services import query_windows
from app.services.embeddings import DinoV2Backend, get_embedding_backend

embedder = get_embedding_backend()
assert isinstance(embedder, DinoV2Backend), (
    "امبدینگ واقعی لازم است: uv sync --group ml. "
    "با بک‌اند قلابی هر عددی که اینجا چاپ شود نویز است."
)

## دو پیکربندی روی یک مجموعه‌ی پرسش

- **کل کادر** — آنچه تا پیش از فاز ۵ اجرا می‌شد: یک بردار از تمام تصویر.
- **هفت پنجره** — تصویر به پنجره‌های هم‌پوشان بریده می‌شود، هر پنجره جست‌وجو
  می‌شود، و هر فرش بهترین امتیازی را که هر پنجره‌ای به او داده نگه می‌دارد.
  کادر کامل خودش یکی از پنجره‌هاست، پس نتیجه نمی‌تواند از حالت قبل بدتر باشد.

In [ ]:
async def measure():
    async with SessionLocal() as session:
        slugs = await retrieval.active_slugs(session)
        depth = await retrieval.catalogue_size(session)
        queries = retrieval.collect_queries(Path.cwd().parent / "data" / "catalog-gen", slugs=slugs)
        whole = await retrieval.evaluate(
            session, queries, embedder=embedder, label="کل کادر", depth=depth
        )
        windowed = await retrieval.evaluate_windowed(
            session, queries, embedder=embedder,
            label=f"×{len(query_windows.DEFAULT_GRID)} پنجره", depth=depth,
        )
        return queries, [whole, windowed]

queries, runs = asyncio.run(measure())
print(f"{len(queries)} پرسش")

In [ ]:
import pandas as pd

pd.DataFrame([run.metrics().as_row() for run in runs])

## به تفکیک نوع عکس

هر سه، عکسی از همان فرش‌اند؛ تفاوتشان این است که فرش چقدر از کادر را می‌گیرد.

In [ ]:
pd.DataFrame(
    [run.metrics(shot=shot).as_row() for shot in retrieval.HELD_OUT_SHOTS for run in runs]
)

In [ ]:
import matplotlib.pyplot as plt

shots = list(retrieval.HELD_OUT_SHOTS)
fig, ax = plt.subplots(figsize=(7, 3.6))
width = 0.36
for offset, run in zip((-width / 2, width / 2), runs):
    ax.bar(
        [i + offset for i in range(len(shots))],
        [run.metrics(shot=s).hit_rate[1] for s in shots],
        width=width,
        label=run.label,
    )
ax.set_xticks(range(len(shots)))
ax.set_xticklabels(shots)
ax.set_ylabel("hit@1")
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("rank-1 retrieval by shot type")
plt.tight_layout()

## Precision@k

با یک قلم مرتبط در هر پرسش این عدد از hit@k مشتق می‌شود و چیز تازه‌ای
نمی‌گوید — اینجا هست چون نقشه‌ی راه با همین نام خواسته بودش.

In [ ]:
pd.DataFrame(
    [
        {"config": run.label, **{f"P@{k}": round(run.metrics().precision_at(k), 4)
                                 for k in retrieval.CUTOFFS}}
        for run in runs
    ]
)

## آنچه شکست می‌خورد

پرسش‌هایی که هنوز رتبه‌ی اول را نمی‌گیرند، برای اینکه در فصل ارزیابی درباره‌شان
حرفی زده شود نه اینکه فقط میانگین گزارش شود.

In [ ]:
worst = sorted(
    (r for r in runs[-1].rankings if r.rank is None or r.rank > 3),
    key=lambda r: (r.rank is None, r.rank or 0),
    reverse=True,
)
pd.DataFrame(
    [{"slug": r.query.slug, "shot": r.query.shot, "rank": r.rank} for r in worst[:15]]
)